# Combined-corpus road-seg retrain — Kaggle, 30 epochs (task A11)

The **from-scratch combined retrain** (DeepGlobe + Massachusetts) at full length —
the definitive version of the shorter local run. No upload needed: both datasets
already live on Kaggle.

## Setup
1. **+ Add Data** → attach **`balraj98/deepglobe-road-extraction-dataset`** *and*
   **`balraj98/massachusetts-roads-dataset`**.
2. **Settings → Accelerator → GPU** (T4 or P100) and **Internet → ON**.
3. **Run all.** Best EMA checkpoint is saved to `/kaggle/working/road_combined_v3.pt`
   — then **Save Version** to keep it, and download it for evaluation/release.

In [ ]:
# 1. Pinned deps (match the repo)
import importlib.util, subprocess, sys
REQ = {"segmentation_models_pytorch": "segmentation-models-pytorch==0.3.4",
       "albumentations": "albumentations==2.0.8", "albucore": "albucore==0.0.24",
       "cv2": "opencv-python-headless==4.10.0.84"}
if any(importlib.util.find_spec(m) is None for m in REQ):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQ.values()])
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 2. Pull the project code (dev branch)
import os, sys, subprocess
REPO = "/kaggle/working/Trace"
if not os.path.exists(REPO + "/src"):
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", "dev",
                           "https://github.com/Akshat-Tiwari69/Trace.git", REPO])
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)
print("repo ready at", REPO)

In [ ]:
# 3. Locate the attached datasets + convert Massachusetts -> DeepGlobe-format
import glob, os
from src.pipeline.p1_segment.build_massachusetts_data import convert_massachusetts

# DeepGlobe: a dir holding *_sat.jpg + *_mask.png
dg_dir = next(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/*_sat.jpg", recursive=True)
              if os.path.exists(p.replace("_sat.jpg", "_mask.png")))
# Massachusetts: a dir whose parent has <split>/ + <split>_labels/
mass_labels = glob.glob("/kaggle/input/**/train_labels", recursive=True)[0]
mass_root = os.path.dirname(mass_labels)
print("DeepGlobe:", dg_dir)
print("Massachusetts root:", mass_root)

mass_out = "/kaggle/working/mass_dg"
convert_massachusetts(os.path.join(mass_root, "train"), mass_labels, mass_out,
                      upsample=2.0, tile_size=512, min_road_fraction=0.005, max_tiles=8000)

In [ ]:
# 4. From-scratch combined retrain (30 epochs, full recipe)
# If you hit CUDA OOM, lower batch_size (e.g. 4). T4/P100 16GB handle 6 fine.
from src.pipeline.p1_segment.train_combined import train_combined, TrainConfig
cfg = TrainConfig(
    data_dirs=[dg_dir, mass_out], out_path="/kaggle/working/road_combined_v3.pt",
    encoder="mit_b3", image_size=512, batch_size=6, lr=2e-4, epochs=30,
    crops_per_image=1, occlusion="heavy", cldice_weight=0.1,
    num_workers=4, device="cuda")
summary = train_combined(cfg)
print("best EMA val IoU:", summary["best_val_iou"])

## After it finishes
- Download `road_combined_v3.pt` from the notebook output (or **Save Version**).
- Hand it back to evaluate **v3 vs v1** on a clean held-out set — release `a4-roadseg-v3`
  **only if it genuinely beats v1** (same honest bar). The local 18-epoch run is the
  quick read; this 30-epoch run is the definitive one.